<a href="https://colab.research.google.com/github/Dennischegemwangi/code-base/blob/main/Building_a_transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bertviz transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 59.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 134.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [bertviz]


In [ ]:
#visualizing attention
from transformers import AutoTokenizer
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model = BertModel.from_pretrained(model_ckpt)

text = "A car needs gas to run"
show(model,"bert",tokenizer,text,display_mode = "light",layer = 0,head = 8)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

100%|██████████| 440473133/440473133 [00:10<00:00, 41019675.30B/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#tokenization
input = tokenizer (text, return_tensors = "pt",add_special_tokens = False)

input.input_ids

tensor([[1037, 2482, 3791, 3806, 2000, 2448]])

In [ ]:
#creating embeddings

from torch import nn
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size,config.hidden_size)

#token_emb

#input embeddings
input_emb = token_emb(input.input_ids)
input_emb.size()

torch.Size([1, 6, 768])

In [ ]:
#creating query,key and value
import  torch
from math import sqrt

query = key = value = input_emb
dim_k = key.size(-1)
att_scores = torch.bmm(query,key.transpose(1,2))/sqrt(dim_k)

In [7]:
#weight creations

import torch.nn.functional as F

weights = F.softmax(att_scores,dim = -1)

weights.sum(dim = -1)


att_outputs = torch.bmm(weights, value)
att_outputs.shape

torch.Size([1, 6, 768])

In [8]:
#scaled dot product attention

def SDPA (query,key,value):
    dim_k = key.size(-1)
    att_scores = torch.bmm(query,key.transpose(1,2))/sqrt(dim_k)
    weights = F.softmax(att_scores,dim = -1)
    return torch.bmm(weights, value)

In [12]:
#multi-head attention implementation

#1 single attention head implementation
class AttentionHead(nn.Module):
      def __init__(self,embed_dim,head_dim):
          super().__init__()
          self.q = nn.Linear(embed_dim,head_dim)
          self.k = nn.Linear(embed_dim,head_dim)
          self.v = nn.Linear(embed_dim,head_dim)
      def foward(self, hidden_state):
          att_outputs = SDPA (self.q(hidden_state),self.k(hidden_state),self.v(hidden_state))

          return att_outputs

In [15]:
#2 multiple attention heads
class multi_head_att(nn.Module):
      def __init__(self,config):

         super().__init__()
         embed_dim = config.hidden_size
         num_heads = config.num_attention_heads
         head_dim = embed_dim//num_heads

         self.heads = nn.ModuleList([AttentionHead(embed_dim,head_dim) for _ in range(num_heads)])
         self.output_linear = nn.Linear(embed_dim,embed_dim)
      def foward (self, hidde_state):
          x = torch.cat([h(hidden_state) for h in self.headss],dim = -1)
          x = self.output_linear(x)


          return x